# BRAMASTRA K8: Two-T4 Cognition and Recursive Improvement Campaign

**Owner-launched experiment.** Run cells top-to-bottom.
Allocation: max **480 elapsed minutes / 960 provisioned GPU-minutes** on two T4 GPUs,
including all phases, retries, evaluations and export.

## Cell groups
1. **Setup** — resolve paths, verify source identity, print the phase plan
2. **Prepare data** — build and validate the generated bundle (idempotent)
3. **E0 gate** — actual CUDA gradients, resume and pilot throughput (required before full run)
4. **Full campaign** — E0 through E6 gated on E0 success
5. **Summarize / Export** — ledger aggregation and result bundle

## Important
- The kernel can be restarted; the SQLite ledger persists the clock.
- If E0 already ran, the full-run cell skips E0 and uses remaining time.
- Stop new training by minute 450; hard stop before 480.
- The old 206-update CPU ledger is untouched.

In [ ]:
# Cell 1: Setup — resolve paths, verify source, print plan
import json, os, sys

REPO = '/kaggle/working/An-Ra-the-new-AGI'
if os.path.isdir(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())

from bramastra_lab.research.runtime.provenance import source_identity
src = source_identity()
print('source identity:', json.dumps(src, indent=2))
import torch
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  cuda:{i}:', torch.cuda.get_device_name(i))
print('Phase plan: E0(0-30) E1(30-150) E2(150-195) E3(195-255) E4(255-315) E5(315-450) E6(450-480)')
print('Stop new training by minute 450. Hard stop before 480.')


## 2. Prepare Data (idempotent)

In [ ]:
# Cell 2: Prepare data bundle
BUNDLE_DIR = '/kaggle/working/bramastra-k8-data'
!python -m bramastra_lab.research.campaigns.k8 prepare --out {BUNDLE_DIR} \n    --training-mechanisms 4096 --controller-mechanisms 256 \n    --development-mechanisms 256 --confirmation-mechanisms 128 \n    --tool-mechanisms 256 --tool-heldout 64 \n    --meta-train 24 --meta-validate 6 --meta-confirm 6


## 3. Validate Data

In [ ]:
# Cell 3: Validate the prepared bundle
!python -m bramastra_lab.research.campaigns.k8 validate --bundle {BUNDLE_DIR}


## 4. E0 Gate (optional standalone)
Run only E0 to prove the CUDA path before committing to the full campaign.
If E0 already ran (receipts exist), this cell is a no-op.

In [ ]:
# Cell 4: E0 gate
RUN_DIR = '/kaggle/working/K8-campaign'
!python -m bramastra_lab.research.campaigns.k8 run --mode e0 \n    --run-dir {RUN_DIR} --data {BUNDLE_DIR} \n    --max-wall-minutes 480 --precision fp16_autocast


## 5. Full Campaign (E0 through E6)
If E0 already ran, this cell skips it and uses remaining time.
**Do not reset the kernel to reset the clock** — the SQLite ledger persists.

In [ ]:
# Cell 5: Full campaign
!python -m bramastra_lab.research.campaigns.k8 run --mode full \n    --run-dir {RUN_DIR} --data {BUNDLE_DIR} \n    --max-wall-minutes 480 --precision fp16_autocast


## 6. Summarize

In [ ]:
# Cell 6: Summarize ledger
!python -m bramastra_lab.research.campaigns.k8 summarize --run-dir {RUN_DIR}


## 7. Export

In [ ]:
# Cell 7: Export result bundle
EXPORT_DIR = '/kaggle/working/K8-results'
!python -m bramastra_lab.research.campaigns.k8 export --run-dir {RUN_DIR} --out {EXPORT_DIR}
print('Export complete. Copy the output directory to persistent storage.')
